# JSBSim C172 Empirical Level-Flight Condition Search v48

`fdm.do_trim(0)` and broad trim grids are not producing a usable fixed-control level-flight state for the current JSBSim C172 setup. This notebook therefore searches empirically for **nearly level fixed-control initial conditions**.

It does not use trim. It simply loops over:

- aircraft model
- initial speed
- initial pitch
- elevator command
- throttle command
- mixture

Then it runs JSBSim with those fixed controls and scores the resulting hold quality.

The best candidate is the one that minimizes altitude drift, vertical speed drift, speed drift, pitch-rate activity, and pitch excursion. This gives a practical initial condition for PID/MPPI comparison even when exact trim is unavailable.

In [1]:
# Colab only. If jsbsim is already installed, this is quick.
try:
    import jsbsim
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'jsbsim', '-q'])
    import jsbsim
print('jsbsim import ok')

jsbsim import ok


In [2]:
import os, time, math, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jsbsim

try:
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB_DRIVE_MOUNTED = True
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)
    COLAB_DRIVE_MOUNTED = False

FPS_PER_KT = 1.687809857
EXPERIMENT_NAME = 'jsbsim_c172_empirical_level_flight_condition_search_v48'
RUN_MODE = 'empirical_hold_grid'
RESULT_ROOT = '/content/drive/MyDrive/Colab Result' if COLAB_DRIVE_MOUNTED else './Colab Result'
SAVE_DIR = os.path.join(RESULT_ROOT, 'PINN_MPC', EXPERIMENT_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)
print('SAVE_DIR:', SAVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SAVE_DIR: /content/drive/MyDrive/Colab Result/PINN_MPC/jsbsim_c172_empirical_level_flight_condition_search_v48


## 1. Search Grid

Start with the coarse grid. If it finds a promising region, set `RUN_FINE_GRID = True` and rerun the fine search cell.

The target is not perfect trim. A useful initial condition should satisfy roughly:

- `abs(alt_drift_ft) < 20 ft` over the hold window
- `abs(tail_vs_fps) < 0.5 ft/s`
- `abs(speed_drift_kts) < 3 kt`
- `q_rms_deg_s < 0.3 deg/s`

If no case meets those thresholds, use the lowest-score candidate as the best practical starting point.

In [3]:
AIRCRAFT_GRID = ['c172p', 'c172x']
INIT_ALT_FT = 3000.0

# Coarse grid. Keep this intentionally manageable.
SPEED_GRID_KTS = np.arange(80.0, 121.0, 5.0)
PITCH_GRID_DEG = np.arange(-4.0, 5.0, 1.0)
ELEVATOR_GRID = np.arange(-0.18, 0.181, 0.03)
THROTTLE_GRID = np.arange(0.45, 0.951, 0.05)
MIXTURE_GRID = [1.0]

DT = 0.02
SETTLE_SECONDS = 2.0
HOLD_SECONDS = 45.0
TAIL_SECONDS = 10.0
TOP_K = 30

# Abort only protects against bad cases. It should not reject ordinary drift.
ALT_ABORT_DELTA_FT = 800.0
PITCH_ABORT_DEG = 25.0
SPEED_ABORT_KTS = 45.0

print('coarse grid size:', len(AIRCRAFT_GRID) * len(SPEED_GRID_KTS) * len(PITCH_GRID_DEG) * len(ELEVATOR_GRID) * len(THROTTLE_GRID) * len(MIXTURE_GRID))

coarse grid size: 23166


In [4]:
def safe_set(fdm, prop, value):
    try:
        fdm[prop] = value
        return True
    except Exception:
        return False


def get_prop(fdm, prop, default=np.nan):
    try:
        return float(fdm[prop])
    except Exception:
        return float(default)


def read_state(fdm):
    return {
        'time_s': get_prop(fdm, 'simulation/sim-time-sec'),
        'h_sl_ft': get_prop(fdm, 'position/h-sl-ft'),
        'h_agl_ft': get_prop(fdm, 'position/h-agl-ft'),
        'vc_kts': get_prop(fdm, 'velocities/vc-kts'),
        'vt_fps': get_prop(fdm, 'velocities/vt-fps'),
        'u_fps': get_prop(fdm, 'velocities/u-fps'),
        'theta_deg': math.degrees(get_prop(fdm, 'attitude/theta-rad', 0.0)),
        'alpha_deg': math.degrees(get_prop(fdm, 'aero/alpha-rad', 0.0)),
        'q_deg_s': math.degrees(get_prop(fdm, 'velocities/q-rad_sec', 0.0)),
        'udot_fps2': get_prop(fdm, 'accelerations/udot-ft_sec2'),
        'wdot_fps2': get_prop(fdm, 'accelerations/wdot-ft_sec2'),
        'qdot_deg_s2': math.degrees(get_prop(fdm, 'accelerations/qdot-rad_sec2', 0.0)),
        'elevator_cmd': get_prop(fdm, 'fcs/elevator-cmd-norm'),
        'elevator_pos': get_prop(fdm, 'fcs/elevator-pos-norm'),
        'throttle_cmd': get_prop(fdm, 'fcs/throttle-cmd-norm'),
        'throttle_pos': get_prop(fdm, 'fcs/throttle-pos-norm[0]'),
        'mixture_cmd': get_prop(fdm, 'fcs/mixture-cmd-norm'),
        'rpm': get_prop(fdm, 'propulsion/engine/propeller-rpm'),
    }


def make_fdm_fixed_ic(aircraft, init_alt_ft, speed_kts, pitch_deg, elevator, throttle, mixture):
    fdm = jsbsim.FGFDMExec(None, None)
    fdm.set_debug_level(0)
    fdm.load_model(aircraft)

    for prop in ['ic/h-sl-ft', 'ic/altitude-ft']:
        safe_set(fdm, prop, init_alt_ft)
    for prop in ['ic/vc-kts', 'ic/vt-kts']:
        safe_set(fdm, prop, speed_kts)
    safe_set(fdm, 'ic/u-fps', speed_kts * FPS_PER_KT)
    safe_set(fdm, 'ic/ubody-fps', speed_kts * FPS_PER_KT)
    safe_set(fdm, 'ic/v-fps', 0.0)
    safe_set(fdm, 'ic/w-fps', 0.0)
    safe_set(fdm, 'ic/p-rad_sec', 0.0)
    safe_set(fdm, 'ic/q-rad_sec', 0.0)
    safe_set(fdm, 'ic/r-rad_sec', 0.0)
    safe_set(fdm, 'ic/phi-deg', 0.0)
    safe_set(fdm, 'ic/theta-deg', pitch_deg)
    safe_set(fdm, 'ic/psi-true-deg', 200.0)
    safe_set(fdm, 'ic/lat-gc-deg', 47.0)
    safe_set(fdm, 'ic/long-gc-deg', -122.0)
    fdm.run_ic()

    # Engine/control state. Set both cmd and pos where available to reduce actuator transient.
    for prop, val in [
        ('fcs/elevator-cmd-norm', elevator),
        ('fcs/elevator-pos-norm', elevator),
        ('fcs/throttle-cmd-norm', throttle),
        ('fcs/throttle-pos-norm[0]', throttle),
        ('fcs/mixture-cmd-norm', mixture),
        ('fcs/mixture-pos-norm[0]', mixture),
        ('propulsion/magneto_cmd', 3),
        ('propulsion/starter_cmd', 1),
        ('fcs/flap-cmd-norm', 0.0),
        ('fcs/flap-pos-deg', 0.0),
        ('gear/gear-cmd-norm', 1.0),
        ('gear/gear-pos-norm', 1.0),
    ]:
        safe_set(fdm, prop, val)
    return fdm


def score_hold(df, init_alt_ft):
    if len(df) < 5:
        return {'valid': False, 'score': np.inf, 'abort': 'EMPTY'}
    t = df['time_s'].to_numpy(dtype=float)
    h = df['h_sl_ft'].to_numpy(dtype=float)
    v = df['vc_kts'].to_numpy(dtype=float)
    theta = df['theta_deg'].to_numpy(dtype=float)
    q = df['q_deg_s'].to_numpy(dtype=float)
    alpha = df['alpha_deg'].to_numpy(dtype=float)
    tail_n = max(3, int(TAIL_SECONDS / DT))
    tail = df.tail(tail_n)
    duration = max(float(t[-1] - t[0]), 1e-6)
    tail_dt = max(float(tail['time_s'].iloc[-1] - tail['time_s'].iloc[0]), 1e-6)
    alt_drift = float(h[-1] - h[0])
    tail_vs = float((tail['h_sl_ft'].iloc[-1] - tail['h_sl_ft'].iloc[0]) / tail_dt)
    speed_drift = float(v[-1] - v[0])
    theta_drift = float(theta[-1] - theta[0])
    q_rms = float(np.sqrt(np.mean(q ** 2)))
    tail_q_rms = float(np.sqrt(np.mean(tail['q_deg_s'].to_numpy(dtype=float) ** 2)))
    h_rmse = float(np.sqrt(np.mean((h - init_alt_ft) ** 2)))
    max_alpha = float(np.max(np.abs(alpha)))
    max_pitch = float(np.max(np.abs(theta)))

    # Weighted practical level-flight score. Lower is better.
    score = (
        1.0 * abs(alt_drift)
        + 15.0 * abs(tail_vs)
        + 3.0 * abs(speed_drift)
        + 15.0 * q_rms
        + 4.0 * abs(theta_drift)
        + 0.25 * h_rmse
        + 0.2 * max_alpha
    )
    feasible = (
        abs(alt_drift) < 20.0
        and abs(tail_vs) < 0.5
        and abs(speed_drift) < 3.0
        and q_rms < 0.3
        and max_pitch < 12.0
        and max_alpha < 10.0
    )
    return {
        'valid': True,
        'feasible_level_hold': bool(feasible),
        'score': float(score),
        'duration_s': duration,
        'alt_drift_ft': alt_drift,
        'alt_rmse_ft': h_rmse,
        'tail_vs_fps': tail_vs,
        'speed_drift_kts': speed_drift,
        'theta_drift_deg': theta_drift,
        'q_rms_deg_s': q_rms,
        'tail_q_rms_deg_s': tail_q_rms,
        'max_abs_pitch_deg': max_pitch,
        'max_abs_alpha_deg': max_alpha,
        'final_alt_ft': float(h[-1]),
        'final_speed_kts': float(v[-1]),
        'final_pitch_deg': float(theta[-1]),
        'abort': 'None',
    }


def run_fixed_hold_case(aircraft, speed_kts, pitch_deg, elevator, throttle, mixture, hold_seconds=HOLD_SECONDS, store_log=False):
    row = {
        'aircraft': aircraft,
        'init_alt_ft': INIT_ALT_FT,
        'init_speed_kts': float(speed_kts),
        'init_pitch_deg': float(pitch_deg),
        'elevator': float(elevator),
        'throttle': float(throttle),
        'mixture': float(mixture),
    }
    rows = []
    abort = 'None'
    try:
        fdm = make_fdm_fixed_ic(aircraft, INIT_ALT_FT, speed_kts, pitch_deg, elevator, throttle, mixture)
        for _ in range(int(SETTLE_SECONDS / DT)):
            safe_set(fdm, 'fcs/elevator-cmd-norm', elevator)
            safe_set(fdm, 'fcs/throttle-cmd-norm', throttle)
            fdm.run()
        h0 = read_state(fdm)['h_sl_ft']
        for _ in range(int(hold_seconds / DT)):
            safe_set(fdm, 'fcs/elevator-cmd-norm', elevator)
            safe_set(fdm, 'fcs/throttle-cmd-norm', throttle)
            fdm.run()
            s = read_state(fdm)
            rows.append(s)
            if abs(s['h_sl_ft'] - h0) > ALT_ABORT_DELTA_FT:
                abort = 'ALT_ABORT'
                break
            if abs(s['theta_deg']) > PITCH_ABORT_DEG:
                abort = 'PITCH_ABORT'
                break
            if s['vc_kts'] < SPEED_ABORT_KTS:
                abort = 'SPEED_ABORT'
                break
        df = pd.DataFrame(rows)
        metrics = score_hold(df, INIT_ALT_FT)
        metrics['abort'] = abort if abort != 'None' else metrics.get('abort', 'None')
        row.update(metrics)
        return row, df if store_log else None
    except Exception as exc:
        row.update({
            'valid': False,
            'feasible_level_hold': False,
            'score': np.inf,
            'abort': f'exception:{type(exc).__name__}',
            'exception_message': str(exc),
        })
        return row, None

## 2. Coarse For-Loop Search

In [5]:
def run_grid_search(aircraft_grid, speed_grid, pitch_grid, elevator_grid, throttle_grid, mixture_grid, label='coarse'):
    rows = []
    best_log = None
    best_key = None
    best_score = np.inf
    total = len(aircraft_grid) * len(speed_grid) * len(pitch_grid) * len(elevator_grid) * len(throttle_grid) * len(mixture_grid)
    print(f'{label} grid size:', total)
    t0 = time.time()
    idx = 0
    for aircraft in aircraft_grid:
        for speed_kts in speed_grid:
            for pitch_deg in pitch_grid:
                for elevator in elevator_grid:
                    for throttle in throttle_grid:
                        for mixture in mixture_grid:
                            idx += 1
                            row, _ = run_fixed_hold_case(aircraft, speed_kts, pitch_deg, elevator, throttle, mixture, store_log=False)
                            row['search_label'] = label
                            rows.append(row)
                            if np.isfinite(row.get('score', np.inf)) and row['score'] < best_score:
                                best_score = row['score']
                                best_key = (aircraft, speed_kts, pitch_deg, elevator, throttle, mixture)
                            if idx % 250 == 0 or idx == total:
                                tmp = pd.DataFrame(rows).sort_values('score')
                                best = tmp.iloc[0]
                                n_feas = int(tmp.get('feasible_level_hold', pd.Series(dtype=bool)).fillna(False).sum())
                                print(f'{label} {idx}/{total} elapsed={time.time()-t0:.1f}s feasible={n_feas} best score={best.score:.2f} {best.aircraft} V={best.init_speed_kts:.1f} pitch={best.init_pitch_deg:.1f} elev={best.elevator:.3f} thr={best.throttle:.2f}')
    df = pd.DataFrame(rows).sort_values('score').reset_index(drop=True)
    if best_key is not None:
        _, best_log = run_fixed_hold_case(*best_key, store_log=True)
    return df, best_log

coarse_df, coarse_best_log = run_grid_search(
    AIRCRAFT_GRID, SPEED_GRID_KTS, PITCH_GRID_DEG, ELEVATOR_GRID, THROTTLE_GRID, MIXTURE_GRID,
    label='coarse'
)

coarse_path = os.path.join(SAVE_DIR, f'empirical_level_hold_coarse_v48_{RUN_MODE}.csv')
coarse_df.to_csv(coarse_path, index=False)
print('Saved:', coarse_path)
print('Top coarse candidates:')
display(coarse_df.head(TOP_K))
print('Feasible count:', int(coarse_df['feasible_level_hold'].fillna(False).sum()))

coarse grid size: 23166


     JSBSim Flight Dynamics Model v1.3.0 Apr  9 2026 10:00:08
            [JSBSim-ML v2.0]

JSBSim startup beginning ...

coarse 250/23166 elapsed=22.3s feasible=0 best score=221.34 c172p V=80.0 pitch=-4.0 elev=0.090 thr=0.95
coarse 500/23166 elapsed=42.7s feasible=0 best score=221.34 c172p V=80.0 pitch=-4.0 elev=0.090 thr=0.95
coarse 750/23166 elapsed=65.0s feasible=0 best score=221.34 c172p V=80.0 pitch=-4.0 elev=0.090 thr=0.95
coarse 1000/23166 elapsed=86.9s feasible=0 best score=221.34 c172p V=80.0 pitch=-4.0 elev=0.090 thr=0.95
coarse 1250/23166 elapsed=108.5s feasible=0 best score=221.34 c172p V=80.0 pitch=-4.0 elev=0.090 thr=0.95
coarse 1500/23166 elapsed=128.9s feasible=0 best score=174.45 c172p V=85.0 pitch=-4.0 elev=0.090 thr=0.80
coarse 1750/23166 elapsed=148.5s feasible=0 best score=174.45 c172p V=85.0 pitch=-4.0 elev=0.090 thr=0.80
coarse 2000/23166 elapsed=171.2s feasible=0 best score=174.45 c172p V=85.0 pitch=-4.0 elev=0.090 thr=0.80
coarse 2250

,aircraft,init_alt_ft,init_speed_kts,init_pitch_deg,elevator,throttle,mixture,valid,feasible_level_hold,score,...,theta_drift_deg,q_rms_deg_s,tail_q_rms_deg_s,max_abs_pitch_deg,max_abs_alpha_deg,final_alt_ft,final_speed_kts,final_pitch_deg,abort,search_label
0,c172p,3000.0,100.0,-4.0,0.15,0.80,1.0,True,False,86.789195,...,0.826624,2.310515,3.433691,3.736191,1.184366,2992.270713,105.249476,-1.587200,None,coarse
1,c172p,3000.0,100.0,-3.0,0.15,0.80,1.0,True,False,92.949230,...,0.014499,2.298717,3.508854,3.079614,1.196395,2988.607035,105.806124,-1.420968,None,coarse
2,c172p,3000.0,100.0,-2.0,0.15,0.80,1.0,True,False,109.998494,...,-0.797213,2.294889,3.585113,2.427519,1.208492,2984.755938,106.362445,-1.252458,None,coarse
3,c172p,3000.0,105.0,-2.0,0.18,0.80,1.0,True,False,121.360171,...,0.289493,1.917033,2.956848,1.804129,0.823964,2950.022831,110.575435,-0.385468,None,coarse
4,c172p,3000.0,100.0,3.0,0.15,0.85,1.0,True,False,124.976481,...,-4.411643,2.853733,4.677509,4.776130,1.268422,3006.339034,112.382913,0.062761,None,coarse
5,c172p,3000.0,105.0,-3.0,0.18,0.80,1.0,True,False,126.328367,...,1.226845,1.913765,2.879201,2.047337,0.813528,2954.061462,109.964199,-0.429604,None,coarse
6,c172p,3000.0,100.0,-1.0,0.15,0.80,1.0,True,False,127.702876,...,-1.609101,2.299198,3.662282,1.787289,1.220645,2980.717169,106.917865,-1.082241,None,coarse
7,c172p,3000.0,100.0,2.0,0.15,0.85,1.0,True,False,128.220457,...,-3.618527,2.828513,4.606585,3.865943,1.256197,3011.559876,111.822759,-0.133906,None,coarse
8,c172p,3000.0,110.0,-4.0,0.18,0.85,1.0,True,False,132.361629,...,0.487163,2.146881,3.301471,3.411320,0.746580,3051.483738,113.046978,-1.552425,None,coarse
9,c172p,3000.0,105.0,-4.0,0.18,0.80,1.0,True,False,134.352011,...,2.164680,1.919444,2.802695,2.636141,0.803124,2957.838340,109.357096,-0.471461,None,coarse


Feasible count: 0


## 3. Fine Search Around Best Coarse Candidate

This narrows the search around the best coarse result. If the coarse result is already good enough, this just refines the actual initial condition for controller experiments.

In [4]:
RUN_FINE_GRID = True

if RUN_FINE_GRID and len(coarse_df):
    b = coarse_df.iloc[0]
    fine_aircraft = [b['aircraft']]
    fine_speed = np.arange(float(b['init_speed_kts']) - 3.0, float(b['init_speed_kts']) + 3.01, 1.0)
    fine_pitch = np.arange(float(b['init_pitch_deg']) - 1.0, float(b['init_pitch_deg']) + 1.01, 0.25)
    fine_elev = np.arange(float(b['elevator']) - 0.025, float(b['elevator']) + 0.0251, 0.005)
    fine_thr = np.arange(float(b['throttle']) - 0.04, float(b['throttle']) + 0.0401, 0.01)
    fine_mix = [float(b['mixture'])]
    fine_df, fine_best_log = run_grid_search(
        fine_aircraft, fine_speed, fine_pitch, fine_elev, fine_thr, fine_mix,
        label='fine'
    )
    fine_path = os.path.join(SAVE_DIR, f'empirical_level_hold_fine_v48_{RUN_MODE}.csv')
    fine_df.to_csv(fine_path, index=False)
    print('Saved:', fine_path)
    print('Top fine candidates:')
    display(fine_df.head(TOP_K))
    print('Fine feasible count:', int(fine_df['feasible_level_hold'].fillna(False).sum()))
else:
    fine_df = pd.DataFrame()
    fine_best_log = None
    print('Fine grid skipped.')

NameError: name 'coarse_df' is not defined

## 4. Select Best Condition and Plot Hold

In [ ]:
if len(fine_df):
    best_df = fine_df.copy()
    best_log = fine_best_log
else:
    best_df = coarse_df.copy()
    best_log = coarse_best_log

best = best_df.iloc[0]
print('Selected empirical level-flight initial condition:')
selected_cols = [
    'aircraft','init_alt_ft','init_speed_kts','init_pitch_deg','elevator','throttle','mixture',
    'score','feasible_level_hold','alt_drift_ft','alt_rmse_ft','tail_vs_fps','speed_drift_kts',
    'q_rms_deg_s','tail_q_rms_deg_s','theta_drift_deg','max_abs_pitch_deg','max_abs_alpha_deg',
    'final_alt_ft','final_speed_kts','final_pitch_deg','abort'
]
for col in selected_cols:
    if col in best.index:
        print(f'{col}: {best[col]}')

selected_path = os.path.join(SAVE_DIR, f'empirical_level_hold_selected_v48_{RUN_MODE}.csv')
pd.DataFrame([best]).to_csv(selected_path, index=False)
print('Saved selected:', selected_path)

if best_log is not None and len(best_log):
    fig, axes = plt.subplots(5, 1, figsize=(11, 11), sharex=True)
    t = best_log['time_s'].to_numpy(dtype=float)
    axes[0].plot(t, best_log['h_sl_ft']); axes[0].axhline(INIT_ALT_FT, color='k', ls='--', lw=0.8); axes[0].set_ylabel('Altitude ft')
    axes[1].plot(t, best_log['vc_kts']); axes[1].set_ylabel('Vc kt')
    axes[2].plot(t, best_log['theta_deg']); axes[2].set_ylabel('Pitch deg')
    axes[3].plot(t, best_log['q_deg_s']); axes[3].set_ylabel('q deg/s')
    axes[4].plot(t, best_log['elevator_cmd'], label='elevator')
    axes[4].plot(t, best_log['throttle_cmd'], label='throttle')
    axes[4].legend(); axes[4].set_ylabel('Controls'); axes[4].set_xlabel('Time s')
    for ax in axes:
        ax.grid(True, alpha=0.3)
    fig.suptitle('Best empirical fixed-control level-flight hold')
    plt.tight_layout()
    fig_path = os.path.join(SAVE_DIR, f'empirical_level_hold_best_v48_{RUN_MODE}.png')
    fig.savefig(fig_path, dpi=160)
    print('Saved plot:', fig_path)
    plt.show()
else:
    print('No best log available.')

## 5. How to Use This Result

Use the selected row as the initial condition for controller comparison:

- `aircraft`
- `init_alt_ft`
- `init_speed_kts`
- `init_pitch_deg`
- initial elevator/throttle command

For PID/MPPI tests, start from this condition and let the controller take over after a short settle period. This reduces false conclusions caused by starting far from any C172 steady-flight condition.